# PicoCal — Cell selection: kNN vs geometric window (notebook 07)

## How big are clusters, really? (the case for cell selection)

*before* deciding how many cells to feed the model, look at the raw distributions. The cell below computes them from the **full non-empty clusters** (only the production-vertex cut `sig_flux_prod_vertex_z < 100`, **no window**), so the whole pipeline is visible and reproducible in the notebook.

Three views: (1) how many cells a cluster has by region, (2) how far each cell sits from the seed in seed-pitch units, and (3) the **decision curve** — how much cluster energy you keep if you take only the *k* nearest cells.

In [1]:
import sys
from pathlib import Path
import numpy as np
import awkward as ak
import uproot

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import derive_geom, PITCH

RKEYS = ["cell_x", "cell_y", "energy", "imodx", "jmody"]
RNAMES = ["R0_15mm", "R1_30mm", "R2_40mm", "R3_60mm", "R4_120mm"]
NFILES = 100
KMAX = 40

files = sorted((repo / "data" / "full").glob("matched_*.root"))[:NFILES]
ncells, region, dist_pitch, dist_region, cover, cover_region = [], [], [], [], [], []
for path in files:
    with uproot.open(path) as f:
        a = f["clusters_matched"].arrays(RKEYS + ["sig_flux_prod_vertex_z"], library="ak")
    vz = ak.to_numpy(a["sig_flux_prod_vertex_z"]).astype(float)
    for i in np.flatnonzero(vz < 100.0):
        c = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in RKEYS}
        if len(c["energy"]) == 0:
            continue
        pitch, mod, rel_x, rel_y, rel_dr, seed = derive_geom(c)
        reg = int(mod[seed])
        ncells.append(len(c["energy"])); region.append(reg)
        dp = rel_dr / pitch[seed]
        dist_pitch.extend(dp.tolist()); dist_region.extend([reg] * len(dp))
        order = np.argsort(rel_dr); es = c["energy"][order]
        cum = np.cumsum(es) / (es.sum() + 1e-9)
        pad = np.ones(KMAX); m = min(len(cum), KMAX); pad[:m] = cum[:m]
        cover.append(pad); cover_region.append(reg)

ncells = np.array(ncells); region = np.array(region)
dist_pitch = np.array(dist_pitch); dist_region = np.array(dist_region)
cover = np.array(cover); cover_region = np.array(cover_region)
present = [r for r in range(len(PITCH)) if (region == r).sum() > 0]
print(f"{len(ncells)} clusters from {len(files)} files")

36852 clusters from 100 files


In [2]:
import pandas as pd

def cov_at(mask, k):
    return round(float(cover[mask][:, k - 1].mean()), 3)

rows = []
for r in present:
    m = region == r; cm = cover_region == r
    rows.append({"region": RNAMES[r], "clusters": int(m.sum()),
                 "median_cells": float(np.median(ncells[m])), "max_cells": int(ncells[m].max()),
                 "E% in 9 (3x3)": cov_at(cm, 9), "E% in 13": cov_at(cm, 13),
                 "E% in 25 (5x5)": cov_at(cm, 25)})
rows.append({"region": "ALL", "clusters": int(len(ncells)),
             "median_cells": float(np.median(ncells)), "max_cells": int(ncells.max()),
             "E% in 9 (3x3)": round(float(cover[:, 8].mean()), 3),
             "E% in 13": round(float(cover[:, 12].mean()), 3),
             "E% in 25 (5x5)": round(float(cover[:, 24].mean()), 3)})
pd.DataFrame(rows)

,region,clusters,median_cells,max_cells,E% in 9 (3x3),E% in 13,E% in 25 (5x5)
0,R0_15mm,6609,352.0,528,0.936,0.952,0.977
1,R1_30mm,7993,137.0,456,0.948,0.961,0.983
2,R2_40mm,9079,81.0,130,0.946,0.960,0.984
3,R3_60mm,11406,36.0,76,0.975,0.984,0.997
4,R4_120mm,1765,9.0,9,1.000,1.000,1.000
5,ALL,36852,81.0,528,0.956,0.968,0.987


In [3]:
import plotly.graph_objects as go

palette = ["#4c78a8", "#f58518", "#54a24b", "#e45756", "#72b7b2"]

maxc = int(np.percentile(ncells, 99)); edges = np.arange(0.5, maxc + 20, 4)
ctr = (edges[:-1] + edges[1:]) / 2
figA = go.Figure()
for r in present:
    h, _ = np.histogram(ncells[region == r], bins=edges)
    figA.add_trace(go.Bar(x=ctr, y=h, name=RNAMES[r], marker_color=palette[r], opacity=0.75))
figA.update_layout(barmode="overlay", template="plotly_white", height=420,
                   title="Non-empty cells per cluster (full cluster, by seed region)",
                   xaxis_title="cells per cluster", yaxis_title="clusters", legend_title="region")
figA.show()

dedges = np.arange(0, 6.01, 0.2); dctr = (dedges[:-1] + dedges[1:]) / 2
figB = go.Figure()
for r in present:
    h, _ = np.histogram(dist_pitch[dist_region == r], bins=dedges, density=True)
    figB.add_trace(go.Bar(x=dctr, y=h, name=RNAMES[r], marker_color=palette[r], opacity=0.6))
figB.update_layout(barmode="overlay", template="plotly_white", height=420,
                   title="Cell distance to seed (seed-pitch units, by region)",
                   xaxis_title="distance / seed pitch", yaxis_title="density", legend_title="region")
figB.add_vline(x=1, line_dash="dot", line_color="#888", annotation_text="3x3 edge")
figB.add_vline(x=2, line_dash="dash", line_color="#888", annotation_text="5x5 edge")
figB.show()

ks = np.arange(1, KMAX + 1)
figC = go.Figure()
for r in present:
    figC.add_trace(go.Scatter(x=ks, y=cover[cover_region == r].mean(0), mode="lines+markers",
                              name=RNAMES[r], line=dict(color=palette[r])))
figC.add_trace(go.Scatter(x=ks, y=cover.mean(0), mode="lines", name="all",
                          line=dict(color="black", width=3, dash="dot")))
figC.add_vline(x=9, line_dash="dot", line_color="#888")
figC.add_vline(x=25, line_dash="dash", line_color="#888")
figC.add_hline(y=0.99, line_dash="dot", line_color="crimson",
               annotation_text="99%", annotation_position="top left")
figC.add_annotation(x=27, y=0.75, showarrow=False, align="left",
                    text="···· vertical line 1:  3x3 = 9 cells<br>– – – vertical line 2:  5x5 = 25 cells",
                    bgcolor="rgba(255,255,255,0.75)", bordercolor="#ccc", borderwidth=1,
                    font=dict(size=12, color="#555"))
figC.update_layout(template="plotly_white", height=460, yaxis_range=[0.6, 1.005],
                   title="Energy captured vs number of nearest cells kept (decision curve)",
                   xaxis_title="nearest cells kept (sorted by distance to seed)",
                   yaxis_title="mean fraction of cluster energy captured", legend_title="region")
figC.show()

### Reading these plots

- **Clusters are large in fine-pitch regions.** R0 (15 mm) has a *median of ~352* non-empty cells (up to 528); R3 (60 mm) ~36; R4 (120 mm) only ~9. Feeding all non-empty cells is hundreds of mostly-empty tokens.
- **Energy concentrates near the seed.** Nearest **9 cells (~3x3) hold ~96%** of cluster energy, **13 ~97%**, **25 (~5x5) ~99%** — the curve is flat after ~13.
- **So a small window is justified.** 9->25 cells buys ~2-3 pp for ~16 noise cells, matching our earlier result that 5x5 slightly *hurt* resolution on clean signal.
- **R4 needs no selection**; R0/R1 are where it matters for the minimum-bias / speed stage.

This motivates the kNN-vs-window comparison below: at a *matched* small budget, does picking cells by grid vs nearest-distance change anything?

## Setup

In [7]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import build, select_window, select_knn, split, resolution, train_eval, Transformer

FILES = 100
EPOCHS = 30
SEEDS = 3
BATCH = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
files = sorted((repo / "data" / "full").glob("matched_*.root"))[:FILES]
{"device": DEVICE, "files": len(files), "epochs": EPOCHS, "seeds": SEEDS}

{'device': 'cuda', 'files': 100, 'epochs': 30, 'seeds': 3}

## The four selection methods

`select_window(c, n)` keeps an n x n grid in seed-pitch units. `select_knn(c, k)` keeps the k cells nearest the seed by distance. We match counts: kNN-9 vs 3x3 (~9 cells), kNN-25 vs 5x5 (~25 cells).

In [8]:
methods = {
    "window_3x3": lambda c: select_window(c, 3),
    "window_5x5": lambda c: select_window(c, 5),
    "knn_9": lambda c: select_knn(c, 9),
    "knn_25": lambda c: select_knn(c, 25),
}
list(methods)

['window_3x3', 'window_5x5', 'knn_9', 'knn_25']

## Build and train each method (multi-seed)

Same clusters and same train/test split across methods (only the cells per cluster differ), so the comparison is fair. Each method is trained over a few seeds; we report mean and std.

In [9]:
def evaluate(selector):
    D = build(files, 5, 100.0, selector=selector)
    y = D["y"]; Et = D["Etrue"]
    ridx = np.flatnonzero(D["region"] == 3)
    rtr, rva, rte = (ridx[s] for s in split(len(ridx)))
    in_dim = D["tok_seed"][int(ridx[0])].shape[1]
    med = int(np.median([D["tok_seed"][i].shape[0] for i in ridx]))
    vals = [train_eval(Transformer(in_dim), D["tok_seed"], y, rtr, rva, rte, Et,
                       EPOCHS, DEVICE, BATCH, seed=s)[0]["sigma_eff"] for s in range(SEEDS)]
    return {"median_cells": med, "mean": round(float(np.mean(vals)), 4), "std": round(float(np.std(vals)), 4)}

results = {}
for name, sel in methods.items():
    results[name] = evaluate(sel)
    print(name, results[name], flush=True)
results

window_3x3 {'median_cells': 9, 'mean': 0.0656, 'std': 0.0109}
window_5x5 {'median_cells': 25, 'mean': 0.0802, 'std': 0.0238}
knn_9 {'median_cells': 9, 'mean': 0.0612, 'std': 0.0038}
knn_25 {'median_cells': 25, 'mean': 0.0733, 'std': 0.0074}


{'window_3x3': {'median_cells': 9, 'mean': 0.0656, 'std': 0.0109},
 'window_5x5': {'median_cells': 25, 'mean': 0.0802, 'std': 0.0238},
 'knn_9': {'median_cells': 9, 'mean': 0.0612, 'std': 0.0038},
 'knn_25': {'median_cells': 25, 'mean': 0.0733, 'std': 0.0074}}

## Comparison

In [10]:
pd.DataFrame([{"method": k, **v} for k, v in results.items()])

,method,median_cells,mean,std
0,window_3x3,9,0.0656,0.0109
1,window_5x5,25,0.0802,0.0238
2,knn_9,9,0.0612,0.0038
3,knn_25,25,0.0733,0.0074


## Verdict

Compare kNN to the grid at matched cell counts. A difference smaller than the std is not real.

In [11]:
def cmp(a, b):
    d = results[a]["mean"] - results[b]["mean"]
    pooled = (results[a]["std"] ** 2 + results[b]["std"] ** 2) ** 0.5
    return {"pair": f"{a} - {b}", "delta": round(d, 4), "sigma": round(d / pooled, 1) if pooled else 0.0,
            "real?": "yes" if pooled and abs(d) >= 2 * pooled else "no"}

[cmp("knn_9", "window_3x3"), cmp("knn_25", "window_5x5"), cmp("window_5x5", "window_3x3")]

[{'pair': 'knn_9 - window_3x3',
  'delta': -0.0044,
  'sigma': -0.4,
  'real?': 'no'},
 {'pair': 'knn_25 - window_5x5',
  'delta': -0.0069,
  'sigma': -0.3,
  'real?': 'no'},
 {'pair': 'window_5x5 - window_3x3',
  'delta': 0.0146,
  'sigma': 0.6,
  'real?': 'no'}]